# Домашнє завдання: Рекомендаційні системи на реальних даних (Goodbooks-10k)

У цьому завданні Ви реалізуєте сучасні (advanced) архітектури рекомендаційних систем із фінального блоку лекції — але вже **не на іграшкових даних, а на реальному датасеті книжкових рейтингів Goodbooks-10k** (десятки тисяч користувачів, тисячі книг, мільйони оцінок).

Це дасть Вам змогу побачити, як підходи поводяться, коли даних справді багато: чому контентних ознак буває замало, як працює retrieval на тисячах елементів, і чому офлайн-метрики на кшталт Recall@K не такі високі, як хотілося б.

**Архітектури, які Ви зберете:** Vector Space Model, Two-Tower, Concat-based ranking (NCF) та двоетапний пайплайн Retrieval → Ranking.

**Стек:** `numpy`, `pandas`, `scikit-learn`, `torch`. GPU не обов'язковий, але з ним тренування буде швидшим (у Colab: *Runtime → Change runtime type → GPU*).

---

## Про датасет

[Goodbooks-10k](https://www.kaggle.com/datasets/zygmunt/goodbooks-10k) — це ~6 млн оцінок 10 000 найпопулярніших книг від 53 424 користувачів. Складається з кількох файлів:

- `ratings.csv` — оцінки: `user_id, book_id, rating` (1–5);
- `books.csv` — метадані книг: `book_id, goodreads_book_id, authors, title, average_rating, ...`;
- `book_tags.csv` — теги/полиці, які користувачі вішали на книги: `goodreads_book_id, tag_id, count`;
- `tags.csv` — розшифровка тегів: `tag_id, tag_name`.

**Важливий нюанс:** на відміну від навчального прикладу, тут **немає готових жанрів**. Жанри доведеться сконструювати самостійно з користувацьких тегів — а це шумні дані (юзери можуть зазначати що завгодно). Це реалістична задача feature engineering, і ми її розберемо в підготовчій частині.

Ще один нюанс із реальних даних: `book_tags.csv` посилається на `goodreads_book_id`, а `ratings.csv` — на `book_id`. Щоб їх поєднати, потрібен джойн через `books.csv`.


## Крок 0. Завантаження даних

Є три способи дістати дані — оберіть будь-який.

**Спосіб A — Kaggle API (рекомендований).** Завантаження з Kaggle API. Зручно, бо декілька файлів і вони завантажаться всі самостійно. Для цього способу завантажте свій `kaggle.json` (Kaggle → Account → Create New API Token), потім виконайте:
```python
from google.colab import files; files.upload()   # оберіть kaggle.json
```
і розкоментуйте відповідний блок нижче.

**Спосіб B — ручне завантаження.** Завантажте архів з посилання на датасет вище з Kaggle, розпакуйте і покладіть `ratings.csv`, `books.csv`, `book_tags.csv`, `tags.csv` поруч із ноутбуком (або через панель Files у Colab).

**Спосіб C — GitHub-дзеркало (фолбек).** Оригінальний автор виклав файли і на GitHub — код нижче підхопить їх автоматично, якщо локально файлів немає.


In [9]:
import os
import numpy as np
import pandas as pd

ratings = pd.read_csv("./data/RecSys Goodbooks/ratings.csv")
books = pd.read_csv("./data/RecSys Goodbooks/books.csv")
book_tags = pd.read_csv("./data/RecSys Goodbooks/book_tags.csv")
tags = pd.read_csv("./data/RecSys Goodbooks/tags.csv")

print("ratings:", ratings.shape)
print("books:  ", books.shape)
print("book_tags:", book_tags.shape, "| tags:", tags.shape)
books[["book_id", "authors", "title", "average_rating"]].head()

ratings: (5976479, 3)
books:   (10000, 23)
book_tags: (999912, 3) | tags: (34252, 2)


,book_id,authors,title,average_rating
0,1,Suzanne Collins,"The Hunger Games (The Hunger Games, #1)",4.34
1,2,"J.K. Rowling, Mary GrandPré",Harry Potter and the Sorcerer's Stone (Harry P...,4.44
2,3,Stephenie Meyer,"Twilight (Twilight, #1)",3.57
3,4,Harper Lee,To Kill a Mockingbird,4.25
4,5,F. Scott Fitzgerald,The Great Gatsby,3.89


## Крок 1. Інженерія жанрів із тегів (feature engineering)

Жанрів у датасеті немає, але є користувацькі теги. Виберемо набір канонічних жанрів і для кожної книги позначимо, які з них їй приписали користувачі. Так ми отримаємо **бінарну матрицю book × genre** — це й будуть контентні ознаки айтемів (аналог `movie_feats_df` із лекції, але здобутий з реальних шумних даних).


In [11]:
# Канонічні жанри, які шукаємо серед тегів
GENRES = ["fantasy", "romance", "mystery", "thriller", "horror", "historical",
          "science-fiction", "young-adult", "nonfiction", "classics",
          "contemporary", "crime"]

# tag_name -> tag_id
name_to_tagid = dict(zip(tags["tag_name"], tags["tag_id"]))
genre_tag_ids = {g: name_to_tagid[g] for g in GENRES if g in name_to_tagid}

# book_tags використовує goodreads_book_id -> мапимо у book_id через books.csv
gid_to_bid = dict(zip(books["goodreads_book_id"], books["book_id"]))
tagid_to_genre = {tid: g for g, tid in genre_tag_ids.items()}

bt = book_tags[book_tags["tag_id"].isin(genre_tag_ids.values())].copy()
bt["book_id"] = bt["goodreads_book_id"].map(gid_to_bid)
bt = bt.dropna(subset=["book_id"])
bt["genre"] = bt["tag_id"].map(tagid_to_genre)

# бінарна матриця book × genre (жанр присутній, якщо користувачі його тегали)
genre_matrix = (
    bt.pivot_table(index="book_id", columns="genre", values="count", aggfunc="sum", fill_value=0)
      .reindex(columns=GENRES, fill_value=0) > 0
).astype(int)

print("Книг із хоча б одним жанром:", (genre_matrix.sum(axis=1) > 0).sum(), "/", len(books))
print("\nРозподіл жанрів:")
print(genre_matrix.sum().sort_values(ascending=False))
genre_matrix.head()

Книг із хоча б одним жанром: 9954 / 10000

Розподіл жанрів:
genre
contemporary       5287
fantasy            4259
romance            4251
mystery            3686
young-adult        3630
classics           2785
historical         2544
thriller           2522
science-fiction    2222
crime              2083
nonfiction         1833
horror             1372
dtype: int64


genre,fantasy,romance,mystery,thriller,horror,historical,science-fiction,young-adult,nonfiction,classics,contemporary,crime
book_id,,,,,,,,,,,,
1,1,1,0,1,0,0,1,1,0,0,1,0
2,1,0,1,0,0,0,0,1,0,1,1,0
3,1,0,0,0,1,0,1,1,0,0,1,0
4,0,0,1,0,0,1,0,1,0,1,1,1
5,0,1,0,0,0,1,0,1,0,1,0,0


## Крок 2. Підвибірка під Colab

6 млн рейтингів — забагато для навчального ноутбука на CPU. Візьмемо **топ-N найпопулярніших книг** і **активних користувачів** (хто поставив ≥ 20 оцінок), а тоді обмежимо число користувачів. Так зберігається щільність взаємодій, а тренування лишається швидким.

> Якщо у Вас GPU або багато часу — сміливо збільшуйте `TOP_BOOKS` та `N_USERS`.


In [23]:
TOP_BOOKS = 3000       # скільки найпопулярніших книг лишити
MIN_USER_RATINGS = 20  # мінімум оцінок на користувача
N_USERS = 4000         # скільки користувачів узяти у підвибірку
LIKE_THRESHOLD = 4     # rating >= 4 вважаємо "лайком" (позитивна взаємодія)

rng = np.random.RandomState(42)

top_books = ratings["book_id"].value_counts().head(TOP_BOOKS).index
r = ratings[ratings["book_id"].isin(top_books)]
active = r["user_id"].value_counts()
r = r[r["user_id"].isin(active[active >= MIN_USER_RATINGS].index)]
sample_users = rng.choice(r["user_id"].unique(), size=min(N_USERS, r["user_id"].nunique()), replace=False)
r = r[r["user_id"].isin(sample_users)].copy()

# лишаємо тільки книги, для яких є жанрові ознаки
r = r[r["book_id"].isin(genre_matrix.index)].copy()

items = sorted(r["book_id"].unique())
users = sorted(r["user_id"].unique())
genre_matrix = genre_matrix.reindex(items).fillna(0).astype(int)

print(f"Взаємодій: {len(r):,} | користувачів: {len(users):,} | книг: {len(items):,}")
print(f"Щільність: {len(r) / (len(users) * len(items)):.4f}")

Взаємодій: 277,903 | користувачів: 4,000 | книг: 1,496
Щільність: 0.0464


In [25]:
import torch
import torch.nn as nn

torch.manual_seed(42)

user_to_idx = {u: i for i, u in enumerate(users)}
item_to_idx = {b: i for i, b in enumerate(items)}
title_of = dict(zip(books["book_id"], books["title"]))

item_feats = torch.tensor(genre_matrix.values, dtype=torch.float32)  # (M, n_genres)
M = len(items)
n_genres = item_feats.shape[1]

# train/val split по взаємодіях
r = r.sample(frac=1, random_state=42).reset_index(drop=True)
n_val = int(len(r) * 0.2)
val_df = r.iloc[:n_val]
train_df = r.iloc[n_val:]

# позитивні пари (лайки) у train
train_pos = train_df[train_df["rating"] >= LIKE_THRESHOLD]
pos_u = torch.tensor([user_to_idx[u] for u in train_pos["user_id"]])
pos_i = torch.tensor([item_to_idx[b] for b in train_pos["book_id"]])

# що користувач уже бачив (щоб не рекомендувати повторно і не семплити як негатив)
from collections import defaultdict
seen_by_user = defaultdict(set)
for u, b in zip(train_df["user_id"], train_df["book_id"]):
    seen_by_user[user_to_idx[u]].add(item_to_idx[b])

# val-лайки для оцінки якості
val_pos = defaultdict(set)
for row in val_df.itertuples():
    if row.rating >= LIKE_THRESHOLD:
        val_pos[user_to_idx[row.user_id]].add(item_to_idx[row.book_id])

print(f"Позитивних пар у train: {len(pos_u):,} | користувачів з val-лайками: {len(val_pos):,}")

Позитивних пар у train: 153,973 | користувачів з val-лайками: 3,956


## Крок 3. Метрика оцінки якості рангування

В лекції ми з вами для оцінки якості використовували **RMSE**. Це валідний варіант, коли треба швидко оцінити якість рек. моделі, але спрощений. RMSE показує, наскільки точно модель передбачає оцінку, яку користувач поставить елементу.

В реальних системах нас ще цікавить **якість ранжування** — наскільки релевантні елементи потрапили в топ списку, який ми реально показуємо користувачу. Для цього використовують ранжувальні метрики: **Precision@K**, **Recall@K**, **NDCG**, **MAP**, **MRR**.

Детальніше можна познайомитись з цими мериками тут:
- огляд метрик для рекомендаційних систем: https://www.evidentlyai.com/ranking-metrics/evaluating-recommender-systems
- Precision та Recall at K: https://www.evidentlyai.com/ranking-metrics/precision-recall-at-k

Нижче давайте реалізуємо функцію `recall_at_k` і будемо оцінювати нею всі наші моделі.

![](https://cdn.prod.website-files.com/660ef16a9e0687d9cc27474a/662c4327f27ee08d3e4d4b2e_6577812c4d677925f1ab5f84_precision_recall_k9.png)

![](https://cdn.prod.website-files.com/660ef16a9e0687d9cc27474a/662c4327f27ee08d3e4d4b47_657781b1f9c868e0cda088f6_precision_recall_k11.png)

**Як працює `recall_at_k`:**

1. Для кожного користувача ми беремо його реальні вподобання з валідаційної вибірки (`val_pos` — книги, які він оцінив на ≥ 4), просимо модель оцінити всі книги й відбираємо топ-K рекомендацій. Перед цим прибираємо книги, які користувач уже бачив у train (щоб не рекомендувати відоме).

2. Далі рахуємо, скільки книг із топ-K справді потрапили в його вподобання (`hits`), і ділимо на загальну кількість релевантних книг (обмежену K, бо більше за K у топ і не влізе).

3. Усереднюємо по всіх користувачах — і отримуємо одне число від 0 до 1: **яку частку того, що користувачу реально сподобалось, модель змогла підняти в топ-K.**

In [28]:
def recall_at_k(score_fn, k=10):
    """Частка val-лайків, що потрапили у топ-k рекомендацій (усереднена по користувачах).
    score_fn(user_idx_tensor) -> матриця оцінок (n_users, M)."""
    eval_users = list(val_pos.keys())
    hits, total = 0, 0
    with torch.no_grad():
        scores = score_fn(torch.tensor(eval_users))  # (len(eval_users), M)
        for row, u in enumerate(eval_users):
            s = scores[row].clone()
            for i in seen_by_user[u]:
                s[i] = -1e9  # прибираємо вже побачене
            topk = torch.topk(s, k).indices.tolist()
            truth = val_pos[u]
            hits += len(set(topk) & truth)
            total += min(len(truth), k)
    return hits / max(total, 1)

---
## Завдання 1. Vector Space Model (векторний підхід)

Перетворимо і книги, і користувачів на вектори в спільному просторі та шукатимемо рекомендації через cosine similarity. Роль ембединга книги відіграє її **нормалізований вектор жанрів** (пояснення про нормалізацію - нижче), а вектор користувача збираємо як **average pooling** ембедингів книг, які він уподобав.

**Що зробити:**

1. Побудуйте `item_emb` — матрицю L2-нормалізованих жанрових векторів усіх книг.
2. Реалізуйте функцію `user_vector(user_idx)` — зважене (за оцінкою) середнє ембедингів уподобаних книг користувача.
3. Реалізуйте функцію `vsm_scores(user_idxs)` — оцінки (cosine) усіх книг для набору користувачів, та порахуйте `recall_at_k`.
4. Покажіть топ-5 рекомендацій для одного користувача (з назвами книг).

**Довідка:**

L2-нормалізація — це ділення вектора на його довжину (L2-норму), щоб отримати вектор тієї ж напрямленості, але одиничної довжини.

Норма рахується як корінь із суми квадратів компонент:

$$\|v\|_2 = \sqrt{(v_1^2 + v_2^2 + \dots + v_n^2)}$$

а сам нормалізований вектор — це
$$\hat{v} = \frac{v}{\|v\|_2}$$

Навіщо це в рекомендаційних системах: після нормалізації **косинусна подібність зводиться до простого скалярного добутку**. Бо $\cos(a, b) = \frac{a \cdot b}{\|a\|\|b\|}$, і якщо обидва вектори вже одиничної довжини, знаменник = 1, тож $\cos(a,b) = a \cdot b$. Це і швидше, і прибирає вплив «довжини» вектора — порівнюється лише напрямок (тобто склад жанрів/смаків), а не те, скільки книг користувач оцінив.

*Приклад:*

Вектор `[3, 4]` має довжину $\sqrt{(9+16)}=5$, після нормалізації стає `[0.6, 0.8]` — напрямок той самий, довжина 1.

In [30]:
# 1. L2-нормалізовані вектори книг
norms    = item_feats.norm(dim=1, keepdim=True).clamp(min=1e-8)
item_emb = item_feats / norms

# Словник user_liked
user_liked = defaultdict(list)
for uid, bid, rating in zip(train_pos["user_id"], train_pos["book_id"], train_pos["rating"]):
    user_liked[user_to_idx[uid]].append((item_to_idx[bid], float(rating)))

# 2. Вектор користувача
def user_vector(user_idx: int) -> torch.Tensor:
    pairs = user_liked[user_idx]
    if not pairs:
        return torch.zeros(n_genres)
    idxs    = torch.tensor([p[0] for p in pairs])
    weights = torch.tensor([p[1] for p in pairs]).unsqueeze(1)
    vecs    = item_emb[idxs]
    vec     = (vecs * weights).sum(dim=0)
    return vec / vec.norm().clamp(min=1e-8)

# 3. Оцінки для набору користувачів (cosine = dot після нормалізації)
def vsm_scores(user_idxs: torch.Tensor) -> torch.Tensor:
    uvecs = torch.stack([user_vector(u.item()) for u in user_idxs])
    return uvecs @ item_emb.T

vsm_recall = recall_at_k(vsm_scores, k=10)
print(f"[VSM] Recall@10 = {vsm_recall:.4f}")

# 4. Топ-5 рекомендацій для першого користувача
demo = 0
s = vsm_scores(torch.tensor([demo]))[0].clone()
for i in seen_by_user[demo]: s[i] = -1e9
top5 = torch.topk(s, 5).indices.tolist()
print(f"\nТоп-5 рекомендацій для user_idx={demo}:")
for rank, idx in enumerate(top5, 1):
    print(f"  {rank}. {title_of.get(items[idx], items[idx])}")

[VSM] Recall@10 = 0.0528

Топ-5 рекомендацій для user_idx=0:
  1. The Clan of the Cave Bear (Earth's Children, #1)
  2. Stardust
  3. The Secret Life of Bees
  4. The Notebook (The Notebook, #1)
  5. Divine Secrets of the Ya-Ya Sisterhood


**Питання:** Recall@10 у векторного підходу досить низький. Чому?


VSM враховує лише жанри книг без додаткових ознак. Саме через це Recall@10 низький

---
## Завдання 2. Two-Tower архітектура

У Завданні 1 вектор користувача рахувався «вручну». Two-Tower натомість **навчає дві окремі башти**: User Tower (з ембединга user_id) та Item Tower (з жанрових ознак). Мережа зводить вектори уподобаних пар близько, а випадкових — далеко. Перевага: вектори книг рахуються один раз і кладуться в індекс (наприклад, FAISS) для швидкого retrieval — рахувати в реальному часі треба лише вектор користувача. Це **late fusion**.

**Що зробити:**

1. Реалізуйте `TwoTower` (user_tower через `nn.Embedding`, item_tower зі жанрових ознак), виходи L2-нормалізуйте.
2. Навчіть на лайках як позитивах і **negative sampling з усього корпусу** (як у пейпері від YouTube) з `BCEWithLogitsLoss` - він є реалізований в PyTorch.
3. Порахуйте `recall_at_k` через попередньо обчислені вектори книг і покажіть приклад рекомендацій.

> **Підказка.** Множте логіти на «температуру» (\~10), бо скалярний добуток нормалізованих векторів лежить у [-1, 1].
> Множення на температуру (\~10) розтягує діапазон логітів до [-10, 10], і тоді сигмоїда може видавати по-справжньому впевнені ймовірності (близькі до 0 і 1). Це дає лосу нормальний градієнт і модель навчається.


In [48]:
EMB_DIM = 32
TEMP    = 10
EPOCHS  = 5
BATCH   = 1024
N_NEG   = 4

N_USERS_COUNT = len(users)

class TwoTower(nn.Module):
    def __init__(self, n_users, n_genres, emb_dim):
        super().__init__()
        self.user_tower = nn.Embedding(n_users, emb_dim)
        self.item_tower = nn.Sequential(
            nn.Linear(n_genres, emb_dim),
            nn.ReLU(),
            nn.Linear(emb_dim, emb_dim),
        )
        nn.init.normal_(self.user_tower.weight, std=0.01)

    def user_emb(self, user_idxs):
        return nn.functional.normalize(self.user_tower(user_idxs), dim=-1)

    def item_emb_fn(self, feats):
        return nn.functional.normalize(self.item_tower(feats), dim=-1)

    def forward(self, user_idxs, item_idxs, all_feats):
        u = self.user_emb(user_idxs)
        i = self.item_emb_fn(all_feats[item_idxs])
        return (u * i).sum(dim=-1) * TEMP

model2 = TwoTower(N_USERS_COUNT, n_genres, EMB_DIM)
opt2   = torch.optim.Adam(model2.parameters(), lr=1e-3)
bce    = nn.BCEWithLogitsLoss()

for ep in range(1, EPOCHS + 1):
    perm = torch.randperm(len(pos_u))
    total_loss, n_batches = 0.0, 0
    for start in range(0, len(pos_u), BATCH):
        idx       = perm[start:start + BATCH]
        bu, bi    = pos_u[idx], pos_i[idx]
        neg_i     = torch.randint(0, M, (len(bu) * N_NEG,))
        all_u     = bu.repeat_interleave(N_NEG + 1)
        pos_neg_i = torch.cat([bi.unsqueeze(1),
                                neg_i.view(len(bu), N_NEG)], dim=1).reshape(-1)
        labels    = torch.zeros(len(all_u))
        labels[::N_NEG + 1] = 1.0

        logits = model2(all_u, pos_neg_i, item_feats)
        loss   = bce(logits, labels)
        opt2.zero_grad(); loss.backward(); opt2.step()
        total_loss += loss.item(); n_batches += 1
    print(f"Epoch {ep}: loss={total_loss/n_batches:.4f}")

# Попередньо обчислюємо всі вектори книг
with torch.no_grad():
    all_item_vecs = model2.item_emb_fn(item_feats)   # (M, D)

def tt_scores(user_idxs):
    with torch.no_grad():
        u = model2.user_emb(user_idxs)
        return (u @ all_item_vecs.T) * TEMP

tt_recall = recall_at_k(tt_scores, k=10)
print(f"\n[Two-Tower] Recall@10 = {tt_recall:.4f}")

demo = 0
s = tt_scores(torch.tensor([demo]))[0].clone()
for i in seen_by_user[demo]: s[i] = -1e9
top5 = torch.topk(s, 5).indices.tolist()
print(f"\nТоп-5 для user_idx={demo}:")
for rank, idx in enumerate(top5, 1):
    print(f"  {rank}. {title_of.get(items[idx], items[idx])}")

Epoch 1: loss=0.5128
Epoch 2: loss=0.4392
Epoch 3: loss=0.4216
Epoch 4: loss=0.4099
Epoch 5: loss=0.4018

[Two-Tower] Recall@10 = 0.0697

Топ-5 для user_idx=0:
  1. The Fountainhead
  2. The Ocean at the End of the Lane
  3. Anansi Boys
  4. Snow Falling on Cedars
  5. Native Son


---
## Завдання 3. Concat-based ranking (NCF)

На відміну від Two-Tower (late fusion), тут **early fusion**: склеюємо ембединг користувача і ознаки книги в один вектор і пропускаємо через MLP, який сам моделює крос-взаємодії. Платою є те, що модель **не можна заіндексувати** — щоб знайти найкращу книгу, треба прогнати кожну пару (user, item). Тому її використовують лише на фінальному ранжуванні кількох кандидатів.

**Що зробити:**

1. Реалізуйте `NCF`: `concat(user_embedding, item_genre_features)` → MLP → один логіт.
2. Навчіть на тих самих позитивах/негативах.
3. Реалізуйте `rank_ncf(user_idx, candidate_idxs)` — ранжування заданого списку кандидатів за `sigmoid` логіта.


In [57]:
class NCF(nn.Module):
    def __init__(self, n_users, n_genres, emb_dim=32):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.mlp = nn.Sequential(
            nn.Linear(emb_dim + n_genres, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )
        nn.init.normal_(self.user_emb.weight, std=0.01)

    def forward(self, user_idxs, item_feat_batch):
        u = self.user_emb(user_idxs)
        x = torch.cat([u, item_feat_batch], dim=-1)
        return self.mlp(x).squeeze(-1)

ncf  = NCF(N_USERS_COUNT, n_genres, emb_dim=32)
opt3 = torch.optim.Adam(ncf.parameters(), lr=1e-3)

for ep in range(1, EPOCHS + 1):
    perm = torch.randperm(len(pos_u))
    total_loss, n_batches = 0.0, 0
    for start in range(0, len(pos_u), BATCH):
        idx       = perm[start:start + BATCH]
        bu        = pos_u[idx]
        bi_pos    = pos_i[idx]
        neg_i     = torch.randint(0, M, (len(bu) * N_NEG,))
        all_u     = bu.repeat_interleave(N_NEG + 1)
        pos_neg_i = torch.cat([bi_pos.unsqueeze(1),
                                neg_i.view(len(bu), N_NEG)], dim=1).reshape(-1)
        labels    = torch.zeros(len(all_u))
        labels[::N_NEG + 1] = 1.0

        logits = ncf(all_u, item_feats[pos_neg_i])
        loss   = bce(logits, labels)
        opt3.zero_grad(); loss.backward(); opt3.step()
        total_loss += loss.item(); n_batches += 1
    print(f"Epoch {ep}: loss={total_loss/n_batches:.4f}")

def rank_ncf(user_idx: int, candidate_idxs: list) -> list:
    """Ранжує candidate_idxs за спаданням NCF-скору."""
    with torch.no_grad():
        u_rep  = torch.tensor([user_idx] * len(candidate_idxs))
        feats  = item_feats[torch.tensor(candidate_idxs)]
        scores = torch.sigmoid(ncf(u_rep, feats))
    ranked = sorted(zip(candidate_idxs, scores.tolist()), key=lambda x: -x[1])
    return [r[0] for r in ranked]

# Recall по NCF (повний прогін по всіх книгах)
def ncf_scores(user_idxs):
    results = []
    with torch.no_grad():
        for u in user_idxs:
            logits = ncf(u.repeat(M), item_feats)
            results.append(torch.sigmoid(logits))
    return torch.stack(results)

ncf_recall = recall_at_k(ncf_scores, k=10)
print(f"\n[NCF] Recall@10 = {ncf_recall:.4f}")

Epoch 1: loss=0.5088
Epoch 2: loss=0.4399
Epoch 3: loss=0.4200
Epoch 4: loss=0.4079
Epoch 5: loss=0.3974

[NCF] Recall@10 = 0.0764


---
## Завдання 4. Двоетапний пайплайн Retrieval → Ranking

Поєднаємо все так, як це працює у великих системах: **Two-Tower швидко відбирає кандидатів** (retrieval серед усіх книг), а **NCF точно ранжує** цю коротку добірку.

**Що зробити:**

1. `retrieve(user_idx, n_candidates)` — топ-N книг за Two-Tower (Завдання 2), без уже побачених.
2. `recommend_pipeline(user_idx, n_candidates, top_k)` — прогнати кандидатів через `rank_ncf` (Завдання 3).
3. Показати для кількох користувачів: що відібрав retrieval і що залишив ranking.


In [66]:
N_CANDIDATES = 100

def retrieve(user_idx: int, n_candidates: int = N_CANDIDATES) -> list:
    """Two-Tower retrieval: топ-N кандидатів без вже побачених."""
    with torch.no_grad():
        u      = model2.user_emb(torch.tensor([user_idx]))
        scores = (u @ all_item_vecs.T * TEMP)[0].clone()
    for i in seen_by_user[user_idx]: scores[i] = -1e9
    return torch.topk(scores, n_candidates).indices.tolist()

def recommend_pipeline(user_idx: int, n_candidates: int = N_CANDIDATES, top_k: int = 10) -> list:
    """Two-Tower → NCF → топ-k."""
    candidates = retrieve(user_idx, n_candidates)
    return rank_ncf(user_idx, candidates)[:top_k]

# Демонстрація
print("Приклад Retrieval → Ranking для трьох користувачів:\n")
for demo in [0, 1, 2]:
    cands = retrieve(demo)
    final = recommend_pipeline(demo)
    print(f"User {demo}:")
    print(f"  Retrieval ({len(cands)} кандидатів) → Ranking, топ-5:")
    for rank, idx in enumerate(final[:5], 1):
        print(f"    {rank}. {title_of.get(items[idx], items[idx])}")
    print()

# Recall пайплайну
def pipeline_scores(user_idxs):
    batch_scores = torch.full((len(user_idxs), M), -1e9)
    for row, u_t in enumerate(user_idxs):
        u     = u_t.item()
        cands = retrieve(u, N_CANDIDATES)
        with torch.no_grad():
            u_rep  = torch.tensor([u] * len(cands))
            feats  = item_feats[torch.tensor(cands)]
            logits = torch.sigmoid(ncf(u_rep, feats))
        for ci, c in enumerate(cands):
            batch_scores[row, c] = logits[ci]
    return batch_scores

pipeline_recall = recall_at_k(pipeline_scores, k=10)
print(f"[Pipeline] Recall@10 = {pipeline_recall:.4f}")

Приклад Retrieval → Ranking для трьох користувачів:

User 0:
  Retrieval (100 кандидатів) → Ranking, топ-5:
    1. Peace Like a River
    2. To Kill a Mockingbird
    3. The Ocean at the End of the Lane
    4. Anansi Boys
    5. Divine Secrets of the Ya-Ya Sisterhood

User 1:
  Retrieval (100 кандидатів) → Ranking, топ-5:
    1. Anansi Boys
    2. The Ocean at the End of the Lane
    3. Peace Like a River
    4. Slaughterhouse-Five
    5. Good Omens: The Nice and Accurate Prophecies of Agnes Nutter, Witch

User 2:
  Retrieval (100 кандидатів) → Ranking, топ-5:
    1. The Ocean at the End of the Lane
    2. Slaughterhouse-Five
    3. Kafka on the Shore
    4. Preludes & Nocturnes (The Sandman #1)
    5. Galápagos

[Pipeline] Recall@10 = 0.0779


**Питання:** навіщо ділити на два етапи, якщо можна ранжувати NCF одразу всі книги?

Якщо ранжувати одразу всі книги, то це займе значно більше часу

---
## Завдання 5. Теоретичний блок (письмові відповіді)

Спираючись на лекцію та на те, що Ви щойно побачили на реальних даних, дайте розгорнуті відповіді в markdown-клітинці нижче.

1. **Чому Recall@10 такий низький?** На реальних даних усі моделі цього ДЗ дають скромний Recall@10. Назвіть щонайменше дві причини (підказки: бідні контентні ознаки — лише 12 жанрів; розрідженість; те, що val-лайки не охоплюють усіх книг, які користувач *міг би* вподобати).
2. **Як покращити якість, не змінюючи архітектуру?** Які додаткові ознаки книг і користувачів з Goodbooks можна було б під'єднати? (автор, рік, середній рейтинг, повний набір тегів через TF-IDF, текстові ембединги опису через BERT...)
3. **Diversity.** Якщо користувач любить фентезі, чому не варто показувати йому 10 фентезі-книг підряд? Як технічно підмішати різноманітність?
4. **Freshness / cold start.** Нова книга має 0 оцінок. Який підхід цього ДЗ зможе рекомендувати її одразу, а який — ні? Чому?
5. **Watch time > CTR (з лекції).** Поясніть, чому YouTube оптимізує час перегляду, а не CTR, і як це технічно вшито у weighted logistic regression.


1. У нас мало ознак і моделі важче відрізняти книги між собою
2. Ми можемо зробити ембединг авторів, розподіл літератури на сучасну і класичну за датою публікації, проаналізувати опис та провести додатковий аналіз оцінок користувачів
3. Якщо ми будемо показувати клієнту тільки фентезі, то у нього з часом пропаде відчуття новизни і в нього не виникне бажання спробувати щось нове завдяки рекомендаціям. Як варіант, ми можемо показувати клієнту популярні зараз книги або гарячі новинки
4. Краще використовувати підходи, які добре справляються із жанровими/текстовими ознаками. Так рекомендації нових книг будуть максимально корисними
5. YouTube оптимізує час перегляду, бо це краще показує рівень задоволення рекомендацією